# 01 — Synthetic Data Generation
**Demand Signal Feature Store — Capstone Project**

This notebook generates two synthetic datasets:
1. **ERP Shipment Data** — Order records with lead times, delays, quantities, costs
2. **Supplier Notes** — Qualitative commentary from procurement, account managers, QA

Both datasets are saved locally and optionally uploaded to Azure Blob Storage.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_generator import generate_erp_data, generate_supplier_notes

## 1. Generate ERP Shipment Data

In [ ]:
erp_df = generate_erp_data(n_records=2000, start_date="2024-01-01", end_date="2025-06-30", seed=42)
print(f"ERP records generated: {len(erp_df)}")
print(f"Date range: {erp_df['order_date'].min()} to {erp_df['order_date'].max()}")
print(f"Suppliers: {erp_df['supplier_id'].nunique()}")
print(f"Products: {erp_df['sku'].nunique()}")
print(f"On-time delivery rate: {erp_df['on_time_delivery'].mean():.1%}")
erp_df.head()

In [ ]:
erp_df.describe()

In [ ]:
# Delay distribution by supplier
erp_df.groupby("supplier_name").agg(
    avg_delay=("delay_days", "mean"),
    max_delay=("delay_days", "max"),
    on_time_rate=("on_time_delivery", "mean"),
    order_count=("order_id", "count"),
).round(2).sort_values("avg_delay", ascending=False)

## 2. Generate Supplier Notes

In [ ]:
notes_df = generate_supplier_notes(erp_df, notes_per_month=3, seed=42)
print(f"Supplier notes generated: {len(notes_df)}")
print(f"Notes with risk signals: {notes_df['has_risk_signal'].sum()} ({notes_df['has_risk_signal'].mean():.1%})")
notes_df.head()

In [ ]:
# Sample risk notes
print("=== Sample Risk Notes ===")
for _, row in notes_df[notes_df["has_risk_signal"]].head(5).iterrows():
    print(f"\n[{row['note_date'].strftime('%Y-%m-%d')}] {row['supplier_name']}:")
    print(f"  {row['note_text']}")

In [ ]:
# Sample normal notes
print("=== Sample Normal Notes ===")
for _, row in notes_df[~notes_df["has_risk_signal"]].head(3).iterrows():
    print(f"\n[{row['note_date'].strftime('%Y-%m-%d')}] {row['supplier_name']}:")
    print(f"  {row['note_text']}")

## 3. Save Locally

In [ ]:
erp_df.to_csv("../data/erp_shipments.csv", index=False)
notes_df.to_csv("../data/supplier_notes.csv", index=False)
print("Saved to ../data/")

## 4. Upload to Azure Blob Storage (Optional)
Uncomment and configure to upload to your Azure Blob container.

In [ ]:
# from azure.storage.blob import BlobServiceClient
# from config.settings import AZURE_STORAGE_CONNECTION_STRING, AZURE_STORAGE_CONTAINER
#
# blob_service = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
# container = blob_service.get_container_client(AZURE_STORAGE_CONTAINER)
#
# # Create container if it doesn't exist
# try:
#     container.create_container()
# except Exception:
#     pass  # already exists
#
# container.upload_blob("raw/erp_shipments.csv", erp_df.to_csv(index=False), overwrite=True)
# container.upload_blob("raw/supplier_notes.csv", notes_df.to_csv(index=False), overwrite=True)
# print("Uploaded to Azure Blob Storage.")